<a href="https://colab.research.google.com/github/Mahnoor-Kalsoom/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mahnoor-Kalsoom/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*
## Unit of Analysis

One row represents the daily search performance of one content item for one pseudonymized client on a specific report date.

The analysis uses the `fact_content_daily_performance` table because it contains daily Google Search Console (GSC) and Google Analytics 4 (GA4) performance metrics for each content item.

## Time Window

For this assignment, I use a mid-panel month (`2025-01`) to avoid using the final month as a development period. This follows the assignment guidance to avoid leakage from the latest month.


In [5]:
!pip -q install duckdb pyarrow

In [6]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE SECRET (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
)
""")

print("Connected successfully!")

Connected successfully!


In [7]:
REL = "hf://datasets/FlyRank/internship-warehouse"

In [8]:
con.sql(f"""
SELECT COUNT(*)
FROM read_parquet(
'{REL}/fact_content_daily_performance/**/*.parquet'
)
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│     78835655 │
└──────────────┘

In [9]:
con.sql(f"""
SELECT *
FROM read_parquet(
'{REL}/fact_content_daily_performance/**/*.parquet'
)
LIMIT 5
""").df()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,True,True,False,30,0,115,...,0,0,0,0,0,0,0,0,0,2025-01
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,True,True,True,False,5,0,358,...,0,0,0,0,0,0,0,0,0,2025-01
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,True,True,True,False,1,0,34,...,0,0,0,0,0,0,0,0,0,2025-01
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714,True,True,True,False,6,0,140,...,0,0,0,0,0,0,0,0,0,2025-01
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,True,True,True,False,5,0,89,...,0,0,0,0,0,0,0,0,0,2025-01


In [10]:
con.sql(f"""
SELECT
    COUNT(*) AS rows,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM read_parquet(
'{REL}/fact_content_daily_performance/**/*.parquet'
)
WHERE month='2025-01'
""").df()

,rows,first_date,last_date
0,1297,2025-01-27,2025-01-31


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Features
- gsc_impressions
- gsc_clicks
- gsc_sum_position
- scroll_events
- sessions_ai

These variables are available before making a content review decision and can be used as model features.

### Label / Proxy

The goal is to rank content items according to their priority for review. A future decline or refresh priority can be used as a proxy label in later modelling stages.

### Context Fields

- report_date
- client_hash_id
- content_hash_id
- month

These identify the observation but are not predictive features.

### Excluded Fields

- client_hash_id
- content_hash_id

These are identifiers only and should not be used as model features because they do not describe content performance and may encourage memorization rather than learning general patterns.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

cols = [
    "report_date",
    "client_hash_id",
    "content_hash_id",
    "gsc_impressions",
    "gsc_clicks",
    "gsc_sum_position",
    "scroll_events",
    "sessions_ai",
    "month"
]

con.sql(f"""
SELECT {",".join(cols)}
FROM read_parquet(
'{REL}/fact_content_daily_performance/**/*.parquet'
)
LIMIT 5
""").df()


,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_sum_position,scroll_events,sessions_ai,month
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,30,0,115,0,0,2025-01
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,5,0,358,0,0,2025-01
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,1,0,34,0,0,2025-01
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714,6,0,140,0,0,2025-01
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,5,0,89,0,0,2025-01


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

The following queries verify the assumptions made in the data contract.

1. Verify the grain (one row represents one content item on one report date).
2. Verify the number of observations and the reporting window.
3. Verify data availability using the required `IS TRUE` condition.

In [16]:
# Query 1 - Grain
print("Query 1")
display(
    con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT report_date || client_hash_id || content_hash_id) AS unique_rows
    FROM read_parquet(
        '{REL}/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2025-01'
    """).df()
)

# Query 2 - Counts and window
print("Query 2")
display(
    con.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS first_day,
        MAX(report_date) AS last_day
    FROM read_parquet(
        '{REL}/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2025-01'
    """).df()
)

# Query 3 - Availability
print("Query 3")
display(
    con.sql(f"""
    SELECT
        COUNT(*) AS available_rows
    FROM read_parquet(
        '{REL}/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2025-01'
      AND gsc_data_available IS TRUE
    """).df()
)

Query 1


,total_rows,unique_rows
0,1297,1297


Query 2


,row_count,first_day,last_day
0,1297,2025-01-27,2025-01-31


Query 3


,available_rows
0,1297


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*


This dataset has several limitations.

The warehouse contains pseudonymized clients and content, so the original websites and page URLs are unavailable. Different clients also have different historical coverage, meaning some clients have shorter observation periods than others. In addition, this analysis only reflects observed search and analytics behaviour and cannot determine why performance changed or establish causal relationships.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

con.sql(f"""
SELECT
MIN(report_date) earliest_date,
MAX(report_date) latest_date,
COUNT(DISTINCT client_hash_id) clients
FROM read_parquet(
'{REL}/fact_content_daily_performance/**/*.parquet'
)
""").df()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,earliest_date,latest_date,clients
0,2025-01-27,2026-06-30,70


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.